<!--
Copyright (c) 2026 OceanBase.

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.
-->

# 05 · 把可核查的工作状态交给接手的人

订单导入器已经能转换两个正常金额，但非法输入与坏行提示还没完成。我们先运行真实检查，把观察保存为 Source，再按官方闭环交接：预览当前工作、提交为制品、接收方按精确版本接续。

**完成后你能做到：** 区分 Work Contract、临时 Prepared Handoff、已提交 Revision、接收回执和 partial 任务结果；知道 `create_artifact` 只是内容已写好时的捷径。

预计 25 分钟。先按 [README](README.md) 安装环境；本篇可以独立运行，不依赖其他 Notebook 的变量或数据。本篇无需模型和 API Key。

按顺序读说明、运行代码，再对照结果。练习可以改输入；完整重跑使用 **Restart Kernel & Run All**。

## 准备本篇实验

这格启动一个回环地址的真实 Server，并创建独立 Scope，默认把数据保存在本篇自己的 SQLite 文件中，也可按 [README](README.md#使用-oceanbase-运行) 显式选择专用 OceanBase 测试库。
`_tutorial.py` 只管理环境和显示结果；下面的业务调用都是可在应用中复用的公开 API。

In [ ]:
import sys
from pathlib import Path

from _tutorial import Tutorial, show, table

from powercontext.http import CreateScopeRequest

if not Path("_tutorial.py").is_file():
    sys.path.insert(0, str(Path.cwd() / "examples" / "jupyter"))


if previous_lab := globals().get("lab"):
    await previous_lab.close()
lab = await Tutorial.start("05", features=())
client = lab.client
assert client is not None

scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 05",
        summary="第 05 篇教程的独立合成数据",
        idempotency_key=f"{lab.run_id}:lesson-05",
    )
)
scope_id = scope.scope_id
show({"title": scope.title, "scope_id": scope_id})

## 1. 先约定这次工作的范围

Work Contract 保存目标、范围与完成条件。本例目标包括金额解析和坏行提示，后面只完成其中一部分，
因此最终结果应为 partial。这里的授权说明描述本篇合成实验范围，不会替代真实应用的权限检查。

In [ ]:
from powercontext.http import CreateWorkContractRequest

contract = await client.create_work_contract(
    CreateWorkContractRequest.model_validate({
        "scope_id": scope_id,
        "source_id": "csv-contract",
        "contract": {
            "schema": "powercontext.work-contract.v1",
            "trust": "untrusted_input",
            "objective": "完成订单 CSV 导入器的金额解析与坏行提示",
            "facts": [],
            "in_scope": ["本篇实验目录中的 money.py 与示例检查"],
            "exclusions": ["真实订单、生产数据库和部署"],
            "completion_criteria": ["正常金额转为整数分", "坏行包含原始行号"],
            "authorization_notes": ["本篇只运行自己创建的示例文件"],
            "open_questions": [],
        },
    })
)
show({"工作约定已保存": True, "Source": contract.source.model_dump(mode="json")})

## 2. 做一段可以重跑的实际工作

在本篇目录写入 `money.py` 和两个正常输入检查，再用独立 Python 进程执行。
结果只能证明这两个输入通过；非法金额与原始行号仍未检查。保留文件摘要供接收方核对。

In [ ]:
import asyncio
import hashlib
import json

project = lab.directory / "csv-project"
project.mkdir()
money_file = project / "money.py"
money_file.write_text(
    "from decimal import Decimal\n\ndef to_cents(text):\n    return int(Decimal(text) * 100)\n",
    encoding="utf-8",
)
check_file = project / "check_money.py"
check_file.write_text(
    "import json\nfrom money import to_cents\n"
    "cases = [('12.34', 1234), ('0.01', 1)]\n"
    "rows = [dict(input=x, expected=y, actual=to_cents(x)) for x, y in cases]\n"
    "print(json.dumps(rows))\n"
    "raise SystemExit(0 if all(r['expected'] == r['actual'] for r in rows) else 1)\n",
    encoding="utf-8",
)


async def run_checks():
    process = await asyncio.create_subprocess_exec(
        sys.executable,
        str(check_file),
        cwd=project,
        stdout=asyncio.subprocess.PIPE,
        stderr=asyncio.subprocess.PIPE,
    )
    stdout, stderr = await process.communicate()
    assert process.returncode == 0, stderr.decode()
    return json.loads(stdout)


checked_rows = await run_checks()
code_digest = hashlib.sha256(money_file.read_bytes()).hexdigest()
table(checked_rows)

## 3. 用统一 Source API 保存实际观察

`create_source` 接收 JSON 内容并返回 Server 生成的 Source ID。接下来再用 `get_source` 读取，
检查保存的内容就是本次实际结果。此时 Source 是证据材料，还没有产生交接制品。

In [ ]:
from powercontext.http import CreateSourceRequest

evidence_content = {
    "file": "money.py",
    "sha256": code_digest,
    "checks": checked_rows,
    "not_checked": ["非法金额", "坏行行号"],
}
evidence = await client.create_source(scope_id, CreateSourceRequest(content=evidence_content))
read_back = await client.get_source(scope_id, "content", evidence.source_id)
assert read_back.content == evidence_content
source_citation = {"kind": "source", "source_ref": {"name": "content", "source_id": evidence.source_id}}
show({"证据 Source": evidence.source_id, "两个实际检查": read_back.content["checks"]})

## 4. 先预览当前工作，不要立刻写成制品

`handoff_current_work` 把已经检查过的边界收成一份临时 Prepared Handoff，供人阅读和确认。
它会保存边界 Source，但**不会**创建 Handoff Artifact。下面先列出制品，确认此时还没有可接续的 Revision。

`state` 引用实际检查记录。`next_action` 只提出未检查项，不声称那些工作已经完成。

In [ ]:
from powercontext.http import HandoffCurrentWorkRequest, ListArtifactsRequest

objective = "完成订单 CSV 导入器的金额解析与坏行提示"
preview = await client.handoff_current_work(
    HandoffCurrentWorkRequest.model_validate({
        "scope_id": scope_id,
        "source_id": "preview-current-state",
        "handoff": {
            "schema": "powercontext.current-work-handoff.v1",
            "trust": "untrusted_input",
            "objective": objective,
            "state": [{"text": "正常金额转换的两个检查通过。", "basis": "verified", "evidence": [source_citation]}],
            "disposition": "continuable",
            "next_action": {"text": "补充非法金额校验和 CSV 坏行原始行号提示。", "basis": "declared", "evidence": []},
            "omissions": ["尚未验证负数、三位小数、非数字和坏行提示。"],
        },
    })
)
before_commit = await client.list_artifacts(scope_id, "handoff", ListArtifactsRequest())
assert before_commit.items == []
assert preview.handoff.content.objective == objective
show({
    "预览目标": preview.handoff.content.objective,
    "已提交 Handoff 数": len(before_commit.items),
    "预览 schema": preview.handoff.model_dump(mode="json", by_alias=True)["schema"],
})

## 5. 确认预览后，再提交为持久 Revision

只有需要留下可接续里程碑时，才把刚才那份 Prepared Handoff 交给 `commit_handoff`。
提交后才会有精确 Artifact 引用；接收方应按这个 Revision 继续，而不是把预览当成已经落库的交接。

In [ ]:
from powercontext.http import CommitHandoffRequest

committed = await client.commit_handoff(CommitHandoffRequest(scope_id=scope_id, handoff=preview.handoff))
handoff_ref = committed.reference
head = await client.get_artifact(scope_id, "handoff", handoff_ref.artifact_id)
assert head is not None and head.revision == handoff_ref.revision == 1
assert head.content["objective"] == objective
show({"已提交交接": handoff_ref.model_dump(mode="json"), "工作状态": head.content["state"]})

## 6. 接收方恢复精确版本，再检查现场

新的 Client 模拟接收方。`continue_handoff` 恢复**已提交**的精确交接；接收方重新运行检查，
确认文件摘要没有变化，然后才记录 accepted 回执。预览对象不能代替这里的 exact Revision。

本篇两个角色使用同一个教学服务和实验目录，展示接续语义；它不是跨用户授权或跨机器传输演示。

In [ ]:
from powercontext.client import PowerContextClient
from powercontext.http import AcknowledgeHandoffRequest, ContinueHandoffRequest

async with PowerContextClient(lab.base_url) as receiver:
    incoming = await receiver.continue_handoff(
        ContinueHandoffRequest(
            scope_id=scope_id,
            selection="exact",
            revision=handoff_ref,
        )
    )
    assert incoming.content is not None
    assert incoming.selected_revision == handoff_ref
    current_rows = await run_checks()
    assert current_rows == checked_rows
    assert hashlib.sha256(money_file.read_bytes()).hexdigest() == code_digest
    receipt = await receiver.acknowledge_handoff(
        AcknowledgeHandoffRequest.model_validate({
            "scope_id": scope_id,
            "source_id": "receiver-checked",
            "receiver": "tutorial-receiver",
            "status": "accepted",
            "selection": "exact",
            "revision": handoff_ref.model_dump(mode="json"),
            "receiver_checks": {"live_state": "confirmed", "capability": "confirmed", "authorization": "confirmed"},
            "message": "已检查示例文件摘要并重跑两个检查；只在本篇实验目录继续。",
        })
    )
show({"接收方恢复的目标": incoming.content.objective, "两个检查重跑通过": True})
show({"回执 Source": receipt.receipt.source.model_dump(mode="json")})

## 7. 交接已接受，任务仍只完成一部分

Task Outcome 记录本次工作实际完成到哪里，并关联接收回执。两个正常输入通过，并不等于金额校验和坏行处理全部完成。
运行之后查看 `partial` 和剩余工作，它们与 accepted 回执描述的是不同事实。

In [ ]:
from powercontext.http import RecordTaskOutcomeRequest

outcome = await client.record_task_outcome(
    RecordTaskOutcomeRequest.model_validate({
        "scope_id": scope_id,
        "source_id": "partial-outcome",
        "outcome": {
            "schema": "powercontext.task-outcome.v1",
            "trust": "untrusted_observation",
            "objective": "完成订单 CSV 导入器的金额解析与坏行提示",
            "status": "partial",
            "summary": "正常金额转换已验证，接收方已确认交接；坏行处理仍待完成。",
            "handoff_receipt_ref": receipt.receipt.source.model_dump(mode="json"),
            "observations": [{"text": "两个正常输入的检查通过。", "basis": "verified", "evidence": [source_citation]}],
            "checks": [
                {
                    "name": "正常金额转换",
                    "status": "passed",
                    "details": "两个输入",
                    "basis": "verified",
                    "evidence": [source_citation],
                }
            ],
            "produced_artifacts": [],
            "remaining_work": ["非法金额校验", "坏行原始行号"],
        },
    })
)
show({
    "任务结果": "partial",
    "剩余工作": ["非法金额校验", "坏行原始行号"],
    "结果 Source": outcome.source.model_dump(mode="json"),
})

## 练习：如果现场已经变化，接收方还能直接接受吗？

下面模拟另一位接收方发现 live_state 不一致。保留 accepted，运行观察 Server 拒绝它。
读完输出后，说明接收方还需要核对或修复什么；不要把未完成的检查改成 confirmed 来消除错误。

In [ ]:
from powercontext.client import ServerResponseError

live_state = "mismatch"
try:
    await client.acknowledge_handoff(
        AcknowledgeHandoffRequest.model_validate({
            "scope_id": scope_id,
            "source_id": "receiver-mismatch",
            "receiver": "another-receiver",
            "status": "accepted",
            "selection": "exact",
            "revision": handoff_ref.model_dump(mode="json"),
            "receiver_checks": {"live_state": live_state, "capability": "confirmed", "authorization": "confirmed"},
        })
    )
except ServerResponseError as error:
    assert error.status_code in (400, 409, 422)
    show({"接收被拒绝": True, "HTTP": error.status_code})
else:
    raise AssertionError("现场不一致时不应接受交接。")

## 捷径：内容已经写好时，可以直接创建 Handoff 制品

官方闭环是预览再 commit。若交接正文已经确定、不需要先给接收方看临时稿，也可以用统一 `create_artifact`。
Handoff 在一个 Scope 内是单例，所以下面换一个 Scope 演示这条捷径，不会改动上面已经提交的 Revision。

In [ ]:
from powercontext.http import CreateArtifactRequest

shortcut_scope = await client.create_scope(
    CreateScopeRequest(
        title="订单 CSV 导入器 · 05 捷径",
        summary="演示直接创建 Handoff",
        idempotency_key=f"{lab.run_id}:handoff-shortcut",
    )
)
shortcut_evidence = await client.create_source(shortcut_scope.scope_id, CreateSourceRequest(content=evidence_content))
shortcut_citation = {"kind": "source", "source_ref": {"name": "content", "source_id": shortcut_evidence.source_id}}
direct = await client.create_artifact(
    shortcut_scope.scope_id,
    CreateArtifactRequest.model_validate({
        "family": "handoff",
        "content": {
            "schema": "powercontext.handoff.v1",
            "objective": objective,
            "state": [{"text": "正常金额转换的两个检查通过。", "citations": [shortcut_citation]}],
            "disposition": "continuable",
            "next_action": {"text": "补充非法金额校验和 CSV 坏行原始行号提示。", "citations": [shortcut_citation]},
            "omissions": [{"text": "尚未验证负数、三位小数、非数字和坏行提示。", "citation": None}],
        },
    }),
)
unchanged_head = await client.get_artifact(scope_id, "handoff", handoff_ref.artifact_id)
shortcut_head = await client.get_artifact(shortcut_scope.scope_id, "handoff", direct.artifact_id)
assert unchanged_head is not None and unchanged_head.revision == handoff_ref.revision
assert shortcut_head is not None and shortcut_head.revision == 1
assert shortcut_scope.scope_id != scope_id
show({
    "捷径 Scope": shortcut_scope.scope_id,
    "捷径制品版本": shortcut_head.revision,
    "主路径版本未变": unchanged_head.revision,
})

## 保存收获，关闭连接

Work Contract 约定目标，Prepared Handoff 只是预览，`commit_handoff` 才留下可接续 Revision。回执记录接收决定，Outcome 记录完成程度。直接 `create_artifact` 是内容已写好时的另一条入口。

下面关闭本篇 Client 和 Server，保留实验文件供检查。中途停止时也可运行这一格；清理方式见 [README](README.md#清理实验数据)。

下一篇：[06](06_reviewed_experience.ipynb)。

In [ ]:
await lab.close()
print("本篇 Server 已关闭。")